# Modelado y Estimación de Límites Humanos en Speedcubing (WCA)
### Análisis Probabilístico con Procesos Gaussianos y Teoría de Valores Extremos (EVT) (2003 – 2026)

Este notebook contiene el pipeline analítico reproducible para investigar la evolución histórica de los **Récords Mundiales Oficiales de la World Cube Association (WCA)** y estimar los **límites asintóticos biológicos y cognitivos** de las 17 disciplinas oficiales, diferenciando adecuadamente la elasticidad de soluciones únicas (*Single*) frente a promedios de consistencia (*Average*).

## 1. Configuración del Entorno y Carga de Librerías

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Estilo visual moderno para las figuras
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Rutas del proyecto
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
CSV_PATH = PROCESSED_DIR / "wca_world_records_progression.csv"
JSON_PATH = BASE_DIR / "outputs" / "limits_summary.json"

print(f"Cargando dataset desde: {CSV_PATH.resolve()}")

## 2. Inspección del Historial de Récords Mundiales WCA

In [ ]:
df_records = pd.read_csv(CSV_PATH, encoding="utf-8")
print(f"Total de registros de Récords Mundiales indexados: {len(df_records)}")
print(f"Categorías encontradas: {df_records['event_id'].unique()}")

# Muestra de la evolución en 3x3x3 Single
df_333_single = df_records[(df_records['event_id'] == '333') & (df_records['type'] == 'single')].sort_values(['date', 'time_sec'])
df_333_single[['date', 'year', 'person_name', 'time_sec', 'competition_id']].tail(10)

## 3. Formateador de Tiempo Oficial WCA
Función para convertir segundos a formato minutero `mm:ss.xx` para marcas superiores a 60 segundos.

In [ ]:
def format_time(val, unit='s'):
    if val is None or np.isnan(val):
        return '--'
    if unit == 's':
        if val >= 60.0:
            minutes = int(val // 60)
            seconds = val % 60
            return f"{minutes}:{seconds:05.2f}"
        return f"{val:.2f}s"
    if unit == 'movs':
        return f"{val:.1f} movs"
    if unit == 'pts':
        return f"{val:.1f} pts"
    return f"{val} {unit}"

print("Ejemplo de formato 90.59s ->", format_time(90.59))
print("Ejemplo de formato 2.76s  ->", format_time(2.76))

## 4. Inferencia Bayesiana: Procesos Gaussianos y Límites con Calibración EVT

In [ ]:
def fit_gp_model(years, records, is_mbf=False, is_single=True):
    years = np.array(years, dtype=float)
    records = np.array(records, dtype=float)
    
    n_pts = len(years)
    current_year = int(years[-1])
    future_years = np.arange(current_year + 1, current_year + 35)
    all_years = np.concatenate([years, future_years])
    
    # Kernel RBF (Squared Exponential)
    length_scale = 10.0
    sigma_f = max(float(np.std(records) * 0.85), 0.1)
    sigma_n = 0.25
    
    dist_k = (years[:, None] - years[None, :]) ** 2
    K = sigma_f**2 * np.exp(-0.5 * dist_k / length_scale**2) + (sigma_n**2) * np.eye(n_pts)
    
    dist_star = (all_years[:, None] - years[None, :]) ** 2
    K_star = sigma_f**2 * np.exp(-0.5 * dist_star / length_scale**2)
    
    dist_star_star = (all_years[:, None] - all_years[None, :]) ** 2
    K_star_star = sigma_f**2 * np.exp(-0.5 * dist_star_star / length_scale**2)
    
    # Inferencia Posterior GP
    K_inv = np.linalg.pinv(K)
    mean_latente = K_star @ K_inv @ (records - np.mean(records)) + np.mean(records)
    cov_latente = K_star_star - K_star @ K_inv @ K_star.T
    raw_std = np.sqrt(np.clip(np.diag(cov_latente), 1e-6, None))
    
    current_rec = float(records[-1])
    initial_rec = float(records[0])
    
    # Estimación del límite por desaceleración empírica y EVT
    if is_mbf:
        asymptotic_mean = current_rec * 1.25
        limit_q025 = current_rec
        limit_q975 = asymptotic_mean * 1.10
        saturation_pct = min(max((current_rec / asymptotic_mean) * 100.0, 50.0), 99.0)
    else:
        mid_idx = n_pts // 2
        r_mid = float(records[mid_idx])
        drop_first_half = max(initial_rec - r_mid, 1e-4)
        drop_second_half = max(r_mid - current_rec, 0.0)
        
        deceleration_ratio = drop_second_half / drop_first_half
        
        if is_single:
            stat_margin_pct = min(max(deceleration_ratio * 0.65, 0.14), 0.22)
            rel_se = 0.055
        else:
            stat_margin_pct = min(max(deceleration_ratio * 0.40, 0.08), 0.14)
            rel_se = 0.040
        
        asymptotic_mean = current_rec * (1.0 - stat_margin_pct)
        limit_q025 = asymptotic_mean * np.exp(-1.96 * rel_se)
        limit_q975 = min(asymptotic_mean * np.exp(1.96 * rel_se), current_rec)
        
        saturation_pct = ((initial_rec - current_rec) / (initial_rec - asymptotic_mean)) * 100.0
        saturation_pct = min(max(saturation_pct, 10.0), 99.5)
        
    # Restricción de monotonía e incertidumbre física acotada
    k_rate = 0.15 if saturation_pct >= 98.0 else (0.18 if saturation_pct >= 95.0 else 0.24)
    corrected_mean = np.zeros_like(mean_latente)
    corrected_std = np.zeros_like(raw_std)
    
    min_so_far = records[0]
    for i in range(n_pts):
        corrected_std[i] = raw_std[i]
        min_so_far = min(min_so_far, records[i])
        corrected_mean[i] = min(mean_latente[i], min_so_far * 1.08)
        corrected_mean[i] = max(corrected_mean[i], asymptotic_mean)
        
    last_hist_val = corrected_mean[n_pts - 1]
    last_hist_std = float(raw_std[n_pts - 1])
    
    for i in range(n_pts, len(all_years)):
        steps_ahead = i - n_pts + 1
        decay_val = asymptotic_mean + (last_hist_val - asymptotic_mean) * np.exp(-k_rate * steps_ahead)
        corrected_mean[i] = max(decay_val, asymptotic_mean)
        corrected_std[i] = min(last_hist_std * (1.0 + 0.06 * steps_ahead), (current_rec - asymptotic_mean) * 0.40)
        
    return {
        "all_years": all_years,
        "historical_years": years,
        "records": records,
        "mean": corrected_mean,
        "std": corrected_std,
        "current_record": current_rec,
        "limit": asymptotic_mean,
        "ci_lower": limit_q025,
        "ci_upper": limit_q975,
        "saturation_pct": saturation_pct
    }

## 5. Visualización: Trayectoria del Récord Mundial en 3x3x3 (Single vs. Average)

In [ ]:
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    summary_data = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for idx, (rtype, title) in enumerate([('single', '3x3x3 Cube (Single)'), ('average', '3x3x3 Cube (Average)')]):
    data = summary_data['events']['333'][rtype]
    years = data['trajectory']['years']
    means = np.array(data['trajectory']['mean'])
    stds = np.array(data['trajectory']['std'])
    hist_recs = data['trajectory']['historical_records']
    limit_val = data['human_limit']['median']
    
    ax = axes[idx]
    # Banda de incertidumbre
    ax.fill_between(years, means - 1.96 * stds, means + 1.96 * stds, color='#3b82f6', alpha=0.15, label='Incertidumbre 95% CI')
    # Trayectoria media
    ax.plot(years, means, color='#2563eb', linewidth=2.5, label='Récord Esperado (GP)')
    # Puntos de récords históricos
    ax.scatter(years[:len(hist_recs)], hist_recs, color='#ef4444', s=35, zorder=5, label='Récords Observados')
    # Límite asintótico
    ax.axhline(limit_val, color='#10b981', linestyle='--', linewidth=2, label=f'Límite Asintótico ({limit_val}s)')
    
    ax.set_title(f"{title}\nSaturación: {data['saturation_index_pct']}% | Límite: {limit_val}s", fontsize=13, fontweight='bold')
    ax.set_xlabel('Año', fontsize=11)
    ax.set_ylabel('Tiempo (segundos)', fontsize=11)
    ax.legend(loc='upper right', frameon=True)
    ax.set_xlim(2003, 2045)

plt.tight_layout()
plt.show()

## 6. Comparativa Global: Resumen de Límites y Saturación en las 17 Categorías WCA

In [ ]:
summary_rows = []

for ev_id, ev_info in summary_data['events'].items():
    name = ev_info['name']
    unit = ev_info['unit']
    for rtype in ['single', 'average']:
        d = ev_info[rtype]
        if not d: continue
        summary_rows.append({
            'Categoría': f"{name} ({rtype.capitalize()})",
            'Récord Actual': format_time(d['current_record'], unit),
            'Límite Estadístico': format_time(d['human_limit']['median'], unit),
            '95% CI': f"[{format_time(d['human_limit']['ci_95_lower'], unit)} - {format_time(d['human_limit']['ci_95_upper'], unit)}]",
            'Llegada al Límite': f"~{d['human_limit']['estimated_convergence_year']}",
            'Saturación (%)': f"{d['saturation_index_pct']}%"
        })

df_summary_table = pd.DataFrame(summary_rows)
df_summary_table

## 7. Gráfico de Barras: Índice de Saturación por Disciplina

In [ ]:
categories = []
saturations = []

for ev_id, ev_info in summary_data['events'].items():
    if ev_info['single']:
        categories.append(ev_info['name'])
        saturations.append(ev_info['single']['saturation_index_pct'])

# Ordenar de menor a mayor saturación
sorted_indices = np.argsort(saturations)
sorted_cats = [categories[i] for i in sorted_indices]
sorted_sats = [saturations[i] for i in sorted_indices]

plt.figure(figsize=(12, 7))
bars = plt.barh(sorted_cats, sorted_sats, color='#3b82f6', edgecolor='#1d4ed8')

for bar, val in zip(bars, sorted_sats):
    plt.text(val - 3.5 if val > 90 else val + 0.5, bar.get_y() + bar.get_height()/2, 
             f"{val:.1f}%", va='center', ha='left', color='white' if val > 90 else '#1e293b', fontweight='bold', fontsize=9)

plt.xlabel('Índice de Saturación (%)', fontsize=11, fontweight='bold')
plt.title('Nivel de Madurez y Proximidad al Límite Asintótico por Disciplina (Single)', fontsize=13, fontweight='bold')
plt.xlim(75, 101)
plt.axvline(98.0, color='#ef4444', linestyle=':', label='Umbral Hiper-Saturado (>98%)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()